necessary libraries

In [3]:
import pandas as pd
from pathlib import Path

defining paths

In [4]:
BASE = Path("/Users/muskaanchugh/Desktop/art_synthetic_data_gen")
SOURCE = BASE / "source_data"
BASELINE = BASE / "baseline_data"
BASELINE.mkdir(exist_ok=True)

loading all source files

In [6]:
adults_line_wise = pd.read_csv(SOURCE / "adults_line_wise.csv")
pediatric_line_wise = pd.read_csv(SOURCE / "pediatric_line_wise.csv")
adult_regimen_wise = pd.read_csv(SOURCE / "adult_regimen_wise.csv")
pediatric_regimen_wise = pd.read_csv(SOURCE / "pediatric_regimen_wise.csv")

phase mapping

In [7]:
def assign_phase(year):
    if year <= 2018:
        return "2015-18"
    elif year <= 2021:
        return "2019-21"
    else:
        return "2022-24"

adults_line_wise["phase"] = adults_line_wise["year"].apply(assign_phase)
pediatric_line_wise["phase"] = pediatric_line_wise["year"].apply(assign_phase)

defining drug-wise patient counts

In [8]:
adult_reg_long = adult_regimen_wise.melt(
    id_vars=["drug", "line"],
    var_name="phase",
    value_name="share"
)

pediatric_reg_long = pediatric_regimen_wise.melt(
    id_vars=["drug", "line"],
    var_name="phase",
    value_name="share"
)

computing counts

In [9]:
# melting line-wise tables to long format
adults_long = adults_line_wise.melt(
    id_vars=["period", "month", "year", "phase"],
    value_vars=["adults_first", "adults_second", "adults_third"],
    var_name="line",
    value_name="count"
)
adults_long["line"] = adults_long["line"].str.replace("adults_", "")

pediatric_long = pediatric_line_wise.melt(
    id_vars=["period", "month", "year", "phase"],
    value_vars=["pediatric_first", "pediatric_second", "pediatric_third"],
    var_name="line",
    value_name="count"
)
pediatric_long["line"] = pediatric_long["line"].str.replace("pediatric_", "")

# merging with regimen shares and compute counts
adults_counts = adults_long.merge(adult_reg_long, on=["phase", "line"])
adults_counts["patient_count"] = adults_counts["count"] * adults_counts["share"]

pediatric_counts = pediatric_long.merge(pediatric_reg_long, on=["phase", "line"])
pediatric_counts["patient_count"] = pediatric_counts["count"] * pediatric_counts["share"]

generating outputs for each drug

In [15]:
def drug_to_filename(drug, population):
    safe = drug.lower().replace("/r", "").replace("+", "_")
    return f"{safe}_{population}_synthetic_baseline.csv"

BASELINE.mkdir(exist_ok=True)

adults_final = adults_counts.groupby(["month", "year", "drug"])["patient_count"].sum().reset_index()
pediatric_final = pediatric_counts.groupby(["month", "year", "drug"])["patient_count"].sum().reset_index()

for drug, df in adults_final.groupby("drug"):
    df[["month", "year", "patient_count"]].rename(columns={"patient_count": "patients"}).assign(
        patients=lambda x: x["patients"].round(0).astype(int)
    ).to_csv(BASELINE / drug_to_filename(drug, "adult"), index=False)

for drug, df in pediatric_final.groupby("drug"):
    df[["month", "year", "patient_count"]].rename(columns={"patient_count": "patients"}).assign(
        patients=lambda x: x["patients"].round(0).astype(int)
    ).to_csv(BASELINE / drug_to_filename(drug, "pediatric"), index=False)